In [45]:
from pathlib import Path
import polars as pl

def non_empty_cols(df: pl.DataFrame) -> list[str]:
    """Columns with at least one non-null value; for strings, also not blank/whitespace-only."""
    string_dtypes = (pl.Utf8, pl.String)
    exprs: list[pl.Expr] = []
    for c in df.columns:
        col = pl.col(c)
        if df.schema[c] in string_dtypes:
            nonempty = col.is_not_null() & (col.str.strip_chars().str.len_chars() > 0)
        else:
            nonempty = col.is_not_null()
        exprs.append(nonempty.any().alias(c))
    flags = df.select(exprs).row(0)
    return [c for c, ok in zip(df.columns, flags) if ok]

data_dir = Path("crazy_gc_data")
csv_paths = sorted(data_dir.rglob("*.csv"))
if not csv_paths:
    raise FileNotFoundError(f"No CSV files under {data_dir.resolve()}")

# Each page can export a different column set → multi-file scan_csv fails with
# "schema lengths differ". Read per file and stack with relaxed alignment.
dfs = [
    pl.read_csv(
        p,
        infer_schema_length=100_000,
        try_parse_dates=True,
    )
    for p in csv_paths
]


In [ ]:
COLUMNS_TO_KEEP = (
    "author.username",
    "author.avatar",
    "attachments.0.url",
    "attachments.1.url",
    "attachments.2.url",
    "attachments.3.url",
    "attachments.4.url",
    "attachments.5.url",
    "attachments.6.url",
    "attachments.7.url",
    "attachments.8.url",
    "reactions.0.emoji.name",
    "reactions.1.emoji.name",
    "reactions.2.emoji.name",
    "reactions.3.emoji.name",
    "reactions.4.emoji.name",
    "reactions.5.emoji.name",
    "referenced_message.content",
    "timestamp",
    
)

reduced_parts: list[pl.DataFrame] = []
for df in dfs:
    present = [c for c in COLUMNS_TO_KEEP if c in df.columns]
    reduced_parts.append(df.select(present))

# Different pages omit columns → different widths. `vertical_relaxed` can still
# error ("schema lengths differ") here; `diagonal` unions columns, null for missing.
reduced_df = pl.concat(reduced_parts, how="diagonal")

print(f"Loaded {len(csv_paths)} file(s), {reduced_df.height} rows × {reduced_df.width} columns")
print(reduced_df.head(5))
reduced_df.write_parquet("discord_gc_combined.parquet")

Loaded 3 file(s), 230073 rows × 19 columns
shape: (5, 19)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ author.us ┆ author.av ┆ attachmen ┆ attachmen ┆ … ┆ reactions ┆ reactions ┆ reference ┆ timestam │
│ ername    ┆ atar      ┆ ts.0.url  ┆ ts.1.url  ┆   ┆ .4.emoji. ┆ .5.emoji. ┆ d_message ┆ p        │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ name      ┆ name      ┆ .content  ┆ ---      │
│ str       ┆ str       ┆ str       ┆ str       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ datetime │
│           ┆           ┆           ┆           ┆   ┆ str       ┆ str       ┆ str       ┆ [μs,     │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ UTC]     │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ rbtro     ┆ 95b91af48 ┆ null      ┆ null      ┆ … ┆ null      ┆ null      ┆ null      ┆ 2025-09- │
│           ┆ 2446975e6 ┆        

# Dataset loading, experimantation and more preprocessing

In [53]:
# ALL attachements to go! They are not accissible since they are private
# Call Parcitipants to go

df = pl.read_parquet("discord_gc_combined.parquet")
print(df.filter(pl.col("author.avatar").is_not_null()).select("author.avatar").to_numpy()[0])
names = df.select("author.username").to_numpy().tolist()
names = [name[0] for name in names]
for name in set(names):
    print(name)


['95b91af482446975e630077533da0f4b']
evandabest_
whorsey.
vhazey
Wordle
joeisaloserandnotgettingoo_34014
kingwooziee
rbtro
plazmama
yannaner
mango2875
oof123


In [ ]:
# All message bodies (non-empty text). Re-run the CSV→parquet cell after adding `content` to COLUMNS_TO_KEEP.
df = pl.read_parquet("discord_gc_combined.parquet")
msgs = (
    df.filter(
        pl.col("referenced_message.content").is_not_null()
        & (pl.col("referenced_message.content").str.strip_chars().str.len_chars() > 0)
    )
    .get_column("referenced_message.content")
)

with open("messages.txt", "w") as f:
    for text in msgs:
        f.write(text + "\n")

u will have sum to lose
expectations
"Soft soft soft soft soft hard"
lmao she unfollowed me on insta
https://tenor.com/view/no-evidence-wash-gif-11390125
To ur hotel
uh sure
Fuck ur country
ahmad almost got cooked
is he dead
im in class
Its not gorey
Ngl this a death that a lot of people are gonna be happy about but nah he didn’t deserve that
There are like 100 ppl that should be shot way before him
Who is that
Is it gore
you need to stop hating
Like there’s something called freedom of speech
Freedom of speech only applies in this country when it’s backing up Israel 😭
Nice rage bate
Antisemitism Awareness Act (H.R. 6090)
    •    Passed by the U.S. House in May 2024, this bill would require scrutinizing antisemitism complaints under Title VI of the Civil Rights Act to consider the IHRA working definition, which includes examples that may label criticism of Israel as antisemitic. Critics argue it could chill free speech, especially on college campuses.  .
    •    The bill stalled in th